In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DSMarket — Task 4: Smart stock replenishment system proposal

<div style="background-color:#2D8C4E;border-left:6px solid #2D8C4E;padding:14px;border-radius:10px">

**Notebook objective**

Translate the forecasting model developed in Task 3 into an operational stock replenishment proposal for DSMarket.

This notebook does not implement a complete production system. Instead, it presents a reasoned and defensible solution to:

- transform daily forecasts into weekly order decisions,
- reduce stockouts through calibration and safety stock,
- and define a minimum deployment, API and monitoring architecture.

</div>

<a id="indice"></a>

## Table of Contents

1. [Context and internal communication](#contexto)
2. [Use case: weekly replenishment](#caso-uso)
3. [Operational replenishment design](#diseno-reposicion)
4. [Required model extensions](#extensiones)
5. [Productization and API](#api)
6. [Executive conclusion](#cierre)
7. 

## How to read this notebook

The notebook follows this sequence:

1. recall the business context and the forecasting result,
2. define the operational replenishment problem,
3. propose how to convert the forecast into a weekly order,
4. identify which technical improvements are needed before production,
5. and close with an executive proposal for business and technology.

The idea is not to build a complete production system here, but to present a realistic operational proposal connected with the project results.

[⬆ Back to table of contents](#indice)

<a id="contexto"></a>

## Context and internal communication

In this task, I move from the predictive layer to the operational layer.

After the previous phases of the project, the relevant conclusion is the following:

- **E2** was the best model from a technical perspective.
- **Calibrated E2** was the best solution from a business perspective, because it corrects underforecasting and reduces stockout risk.

On that basis, this notebook proposes how to use that result to support smarter weekly replenishment.

[⬆ Back to table of contents](#indice)

## Section 1 — Internal communication

**From:** Nicole Chen, Senior Data Scientist  
**To:** Paul Rogers, CFO  
**CC:** Michelle Huggins, CDO  
**Subject:** Proposal for a forecasting-based weekly replenishment system — DSMarket  
**Date:** March 2025

---

Paul,

As agreed, I am presenting the proposal to apply the forecasting models developed in the previous phases of the project to stock replenishment in the New York, Boston and Philadelphia stores.

Over the past few weeks, three work blocks were completed:

- **EDA and time series** — Demand patterns were identified by store, city and category, along with segments with higher operational risk and impact.
- **Clustering** — Products were segmented into groups with different demand behaviors, useful as a contextual analysis layer.
- **Forecasting** — A prediction system was built where **E2** was the best technical option and **calibrated E2** was the best operational option, as it corrected the underforecasting bias detected during the test period.

Based on these results, this proposal details:

1. how to translate daily predictions into weekly replenishment orders,
2. what additional elements are needed to reduce stockouts,
3. which technical extensions should be addressed before a robust deployment,
4. and how to expose the solution through an API and monitoring.

I remain available to review this proposal before the joint presentation with business and technology.

Nicole

[⬆ Back to table of contents](#indice)

<a id="caso-uso"></a>

## Use case: weekly replenishment

The objective of this proposal is to transform forecasting into a useful operational decision.

The question is no longer which model is best in purely predictive terms, but:

**how to use the prediction to reduce stockouts without generating unnecessary excess inventory.**

[⬆ Back to table of contents](#indice)

### 2.1 The operational problem

DSMarket currently manages stock replenishment manually or through fixed rules, for example replenishing when stock falls below a predefined threshold.

This approach generates two recurring problems:

- **Stockouts** in high-demand or volatile-demand products, with direct sales loss and deterioration of customer experience.
- **Excess stock** in slow-moving or seasonal products, with storage costs, capital immobilization and higher risk of obsolescence or expiration.

The project opportunity is to replace that approach with one based on demand forecasting and explicit operational risk control.

The idea is not to order “the same as we expect to sell,” but to build an order recommendation that combines:

- expected demand,
- model bias correction,
- safety stock,
- and available stock.

[⬆ Back to table of contents](#indice)


### 2.2 Proposal objective

The smart replenishment proposal pursues four business objectives:

1. **Reduce stockouts** in products and stores with higher operational risk.
2. **Reduce excess inventory** in low-turnover or seasonal items.
3. **Replace fixed rules with quantitative decisions**, supported by forecasting and observed real error.
4. **Prepare a productizable foundation**, consumable by business teams and internal systems.

In practical terms, this proposal aims to answer a very simple question:

**How much should each store order for each product next week, given expected sales and the risk of falling short?**

[⬆ Back to table of contents](#indice)

<a id="diseno-reposicion"></a>

## Operational replenishment design

In this section, I translate the forecasting result into a concrete order logic.

The central idea is that a useful business forecast should not be converted directly into a purchase order,
but should pass through an additional layer of correction and protection against operational risk.

[⬆ Back to table of contents](#indice)

### 3.1 Proposed operational logic

The proposal does not consist of directly converting the prediction into a purchase order.

To reduce stockout risk, replenishment should be built as a combination of three elements:

1. **Expected demand for the next 7 days**, obtained from the aggregated daily forecast.
2. **Safety stock**, calculated based on the model's historical error and the desired service level.
3. **Available in-store stock**, subtracted at the time the order is generated.

In this way, the system stops relying on fixed rules or arbitrary thresholds
and becomes based on a quantitative forecast adjusted to real demand behavior.

The important decision here is that replenishment no longer responds only to how much is expected to be sold,
but also to how much risk the business is willing to accept.

[⬆ Back to table of contents](#indice)

### 3.2 From daily prediction to weekly order

The model generates predictions at **store × product × day** level.  
For weekly replenishment, these predictions are translated into operational demand in two steps.

**Step 1 — Weekly aggregation of the daily forecast**

For each store × product combination, the 7 days of the horizon are summed:

`Estimated_demand_7d = Σ daily_prediction_i (i = 1..7)`

**Step 2 — Calibration application**

Because the E2 model showed systematic underforecasting during the test period,  
a global calibration factor obtained from the predicted/actual ratio is applied:

`Calibrated_demand_7d = Estimated_demand_7d × 1.3657`

This amount represents the best operational estimate of expected weekly demand.

The important decision here is that replenishment is not based on the raw model prediction,
but on its calibrated version, which reduces the risk of under-supply.

[⬆ Back to table of contents](#indice)

### 3.3 Safety stock and service level

Even with calibration, the prediction remains subject to uncertainty.

For that reason, the proposal incorporates a **safety stock** layer that acts as a buffer against:

- residual model error,
- real demand variability,
- and the risk of falling short on critical items.

The safety stock logic should depend on two elements:

1. **Product or segment variability**
2. **Target service level**

This makes it possible to adjust replenishment aggressiveness according to business context.

Examples of target service level:

- **90%** → lower safety stock, higher inventory efficiency
- **95%** → reasonable balance between cost and risk
- **99%** → conservative approach for critical products

The key implication is that the forecast does not replace safety stock.
It complements it.

[⬆ Back to table of contents](#indice)

### 3.4 Proposed safety stock formula

A simple and defensible way to build safety stock is to base it on the model's historical error:

`Safety_stock = z × σ_error × sqrt(L)`

Where:

- **z** represents the desired service level,
- **σ_error** represents the standard deviation of historical error,
- **L** represents the lead time or coverage period considered.

In an initial version of the system, this calculation can be done at the level of:

- category,
- product cluster,
- or store × category,

depending on which level of granularity is more stable.

The advantage of this approach is that safety stock stops being an arbitrary rule
and becomes anchored in the real behavior of the model error.

[⬆ Back to table of contents](#indice)

### 3.5 Final formula for the recommended order

The proposed operational logic for each store × product combination is:

`Recommended_order = max(0, Calibrated_demand_7d + Safety_stock - Available_stock)`

Where:

- **Calibrated_demand_7d** captures the best operational sales estimate for the following week.
- **Safety_stock** protects against model uncertainty and demand variability.
- **Available_stock** represents the usable inventory at the time the order is generated.

This formulation correctly separates two different problems:

- **forecasting**, which estimates expected demand,
- and **replenishment**, which also incorporates risk tolerance and the target service level.

From a business perspective, this is the key piece of the proposal:
the order is no longer built on intuition or fixed rules,
but on an explicit combination of forecast, error and available stock.

[⬆ Back to table of contents](#indice)

📌 **Conclusion / Decision**

The replenishment proposal does not use the daily prediction directly,
but transforms it into a calibrated weekly estimate protected with safety stock.

This makes it possible to convert the forecasting result into a more robust operational decision,
especially in a context where model underforecasting would generate stockouts
if used without additional correction.

[⬆ Back to table of contents](#indice)

<a id="extensiones"></a>

## Required model extensions

The proposed system can be used in an initial pilot with **calibrated E2**.

However, before a robust large-scale deployment, several improvements should be incorporated
to increase accuracy, reduce bias and make the solution more stable from an operational perspective.

[⬆ Back to table of contents](#indice)


### 4.1 Identified technical debt

The calibrated E2 model is functional and validated, but it was designed as a forecasting solution applied in a notebook environment.

Before using it as a production stock replenishment system, we identify four specific improvements that would increase its robustness:

1. **more appropriate handling of zero inflation**,  
2. **greater sensitivity to series identity**,  
3. **finer calibration than the current global factor**,  
4. **ability to estimate uncertainty, not only point predictions**.

These improvements are not essential for a limited pilot,
but they are recommended for a stronger operational deployment.

[⬆ Back to table of contents](#indice)

### 4.2 E5 — Tweedie objective for zero inflation

**56.3%** of the dataset records have sales equal to zero.

The current model (`objective = regression_l1`) optimizes the median and tends to behave conservatively,
which helps explain part of the underforecasting bias observed before calibration.

A natural improvement would be to train a variant with a **Tweedie objective**,
a distribution designed for data with excess zeros and positive values.

### Expected impact

- better fit on low-turnover series,
- lower need for global post-processing correction,
- and greater consistency between the target distribution and the model objective function.

### Practical reading

This improvement does not replace calibration by itself,
but it could reduce the model's structural bias during training.

[⬆ Back to table of contents](#indice)

### 4.3 E6 — Series identity as a feature

The current model does not explicitly incorporate the identity of the series it is predicting.

Adding categorical variables such as:

- `store_code`
- `category`
- `city`

would allow the model to learn structural differences between demand contexts.

This is especially relevant because Task 3 documented different bias by category:

- **ACCESSORIES** → more negative bias
- **HOME & GARDEN** → intermediate bias
- **SUPERMARKET** → less severe bias, but still relevant

### Expected impact

- lower bias by segment,
- better fit to heterogeneity across stores and categories,
- and less dependence on a uniform global calibration.

The hypothesis here is simple:
part of the current error does not come only from the sales level,
but from the fact that the model does not distinguish the context it is predicting well enough.

[⬆ Back to table of contents](#indice)

### 4.4 E7 — Calibration by category

The current calibration factor (**×1.3657**) is applied globally to all products and stores.

This is useful as a first operational correction,
but it oversimplifies a reality where bias varies significantly by category.

The most immediate and lowest-cost improvement is to calculate an independent calibration factor for each category.

### Estimated correction factors

| Category | Approximate previous bias | Estimated correction factor |
|---|---:|---:|
| ACCESSORIES | more negative | ×1.82 |
| HOME & GARDEN | intermediate | ×1.51 |
| SUPERMARKET | less severe | ×1.27 |

### Expected impact

- finer bias correction,
- lower underforecasting in the most problematic segments,
- and less dependence on a single global multiplier.

### Practical reading

This is the extension with the best impact-to-effort ratio.

For that reason, if only one improvement had to be prioritized before the first pilot deployment,
the recommendation would be to **implement E7 first**.

[⬆ Back to table of contents](#indice)

### 4.5 E8 — Prediction intervals and quantiles

For replenishment with controlled risk, a point prediction is not always enough.

From a business perspective, it is useful to choose between different coverage scenarios, for example:

- a more cost-aggressive replenishment,
- a balanced replenishment,
- or a more conservative replenishment to avoid stockouts.

One way to do this is to train **quantile regression** models that return several prediction levels:

- **Q10** → optimistic scenario
- **Q50** → central prediction
- **Q90** → conservative scenario

### Expected impact

- ability to translate forecasting into explicit service levels,
- better alignment between analytics and inventory decisions,
- and a more flexible replenishment policy according to category or criticality.

### Practical reading

This extension is especially valuable,
but it is also more demanding in design and validation than the previous ones.

For that reason, it is proposed as a medium-term improvement.

[⬆ Back to table of contents](#indice)

### 4.6 Extension prioritization

| Extension | Expected impact | Estimated effort | Priority |
|---|---|---|---|
| E7 — Calibration by category | High | Low | **Immediate** |
| E5 — Tweedie objective | High | Medium | Short term |
| E6 — Series identity | Medium | Medium | Short term |
| E8 — Quantiles / intervals | High | High | Medium term |

### Recommendation

The most realistic path for DSMarket would be:

1. **Initial pilot with calibrated E2**
2. **Immediate implementation of E7**
3. **Subsequent experimentation with E5 and E6**
4. **Design of a quantile-based solution in a later phase**

In this way, the company can capture operational value early
without giving up a progressive technical improvement of the system.

[⬆ Back to table of contents](#indice)

📌 **Conclusion / Decision**

The proposed solution can be used as the basis for a pilot,
but a robust large-scale deployment requires reducing the model's structural bias
and improving sensitivity to the context of each series.

The most immediate and cost-effective improvement would be **calibration by category**,
while Tweedie, series identity and quantiles form the natural roadmap for technical evolution.

[⬆ Back to table of contents](#indice)

<a id="api"></a>

## Productization and API

A replenishment proposal is not useful if it depends on manual execution in a notebook.

For that reason, this section translates the analytical logic into a minimum operational architecture,
designed so that business teams and internal systems can consume the order recommendation
without depending on the technical detail of the model.

[⬆ Back to table of contents](#indice)


### 5.1 Overview

Moving from a model trained in a notebook environment to an operational solution requires three components:

1. **Update and retraining pipeline**, to incorporate recent data.
2. **Prediction API**, to expose the order recommendation as a service.
3. **Monitoring system**, to detect model degradation and trigger recalibration or review.

These three components form the minimum skeleton of an MLOps solution for DSMarket.

The idea is not to deploy a complex platform from day one,
but to define a progressive and realistic architecture that allows the pilot to move into operation.

[⬆ Back to table of contents](#indice)

### 5.2 Periodic retraining pipeline

The model should be updated to incorporate the most recent demand patterns
and prevent the solution from losing quality over time.

### Proposed cadence

| Process | Frequency | Detail |
|---|---|---|
| Full retraining | Monthly | New model with a 2-year sliding window |
| Recalibration | Monthly | Update global or category-level factor |
| Feature update | Weekly | Incorporate latest real sales and exogenous variables |
| Automatic validation | At each retraining | MAE and bias over the last 4 weeks |

### New model rejection criterion

If the retrained model materially worsens compared with the model in production,
the pipeline should not promote it automatically.

A simple and defensible rule would be:

- reject the new model if **MAE** worsens by more than **10%**
- or if **bias** moves significantly away from the acceptable range observed in production

In that case, the system keeps the previous version and generates an alert for review.

[⬆ Back to table of contents](#indice)

### 5.3 Recommended pipeline stack

For a first operational version, the recommended architecture would be:

- **Orchestration:** Apache Airflow or Prefect
- **Model registry:** MLflow
- **Data storage:** DSMarket corporate warehouse
- **Containerization:** Docker
- **Scheduled execution:** cloud environment or internal server with scheduler

The specific choice among these tools does not change the system logic.
What matters is that the solution allows the team to:

- version models,
- repeat training in a controlled way,
- and preserve traceability of metrics and promotion decisions.

[⬆ Back to table of contents](#indice)

### 5.4 API design (Martin's requirement)

The API exposes predictions as a REST service,
so that any internal system can query order recommendations
without depending on the technical detail of the model.

### Main endpoint

`POST /api/v1/forecast`

### Example request

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "horizon_days": 7,
  "service_level": 0.95
}
```

--- JSON ---
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "forecast_units_7d": 140,
  "safety_stock": 35,
  "recommended_order": 175,
  "model_version": "E2_calibrated_v1",
  "confidence": "high"
}


### 5.5 Secondary endpoints

| Endpoint | Method | Description |
|---|---|---|
| `/api/v1/forecast/batch` | POST | Prediction for multiple products |
| `/api/v1/forecast/store/{store_id}` | GET | Recommended orders for a full store |
| `/api/v1/model/metrics` | GET | Current metrics of the model in production |
| `/api/v1/model/status` | GET | Active version, last retraining date and status |

### Recommended stack

- **API framework:** FastAPI
- **Containerization:** Docker
- **Deployment:** AWS or Azure
- **Authentication:** internal token or corporate gateway

The recommendation here is not to build a complex architecture from the start,
but a small, clear and maintainable API.

[⬆ Back to table of contents](#indice)

### 5.6 Monitoring and alerts

Once deployed, the model must be monitored continuously.

The most important metric for this use case is not only the average error,
but the drift between real demand and prediction, because sustained underforecasting
translates directly into stockout risk.

### Key metrics to monitor

| Metric | Frequency | Indicative alert threshold |
|---|---|---|
| Global predicted/actual ratio | Weekly | Outside 0.90 – 1.10 range |
| Category-level ratio | Weekly | Outside 0.85 – 1.15 range |
| 4-week rolling MAE | Weekly | Increase > 15% over baseline |
| % real stockouts | Weekly | > 5% of store × product combinations |

### Recommended operational rule

If the global or category-level ratio deviates persistently for several weeks,
the system should trigger an alert and propose recalibration or model review.

This prevents treating as noise a problem that can quickly turn
into lost sales or overstock.

[⬆ Back to table of contents](#indice)

### 5.7 Recalibration and maintenance

Calibration should not be understood as a one-time, immutable adjustment.

In a real environment, the correction factor should be reviewed periodically,
because it may change if there are changes in:

- product mix,
- promotional intensity,
- category behavior,
- or the base quality of the model.

### Practical recommendation

The first version of the system can work with:

- **calibrated E2**
- monthly recalibration
- and weekly monitoring of the predicted/actual ratio

In a later phase, this logic should evolve toward:

- category-level calibration,
- automatic stability validation,
- and eventually quantile models to support decisions by service level.

[⬆ Back to table of contents](#indice)

📌 **Conclusion / Decision**

The proposed solution does not need a complex platform to start,
but it does need a minimum structure that allows the model to be updated, the order recommendation to be exposed, and its production quality to be monitored.

The combination of a periodic pipeline, lightweight API and continuous monitoring
is enough to turn the analytical work into a defensible operational solution for DSMarket.

[⬆ Back to table of contents](#indice)

<a id="cierre"></a>

## Executive conclusion

In this final section, I synthesize the proposal from the business perspective and translate the previous technical work into a clear operational recommendation for DSMarket.

[⬆ Back to table of contents](#indice)

### 6.1 Summary for Paul Rogers, CFO

This document presents a proposal to convert the forecasting work developed in Tasks 1 to 3 into an operational stock replenishment solution for DSMarket.

### What exists today

- A forecasting solution validated over a 28-day horizon.
- A model where **E2** was the best technical option.
- A **calibrated E2** version that corrects underforecasting bias and is more suitable for operational use.
- Full coverage across the **store × product** combinations of the analyzed business.

### What this proposal enables

- Automatic weekly order calculation for each store × product combination.
- Use of **safety stock** based on real model error, not arbitrary rules.
- Ability to define different service levels according to risk, category or operational context.
- System exposure through an API, consumable by other areas or internal systems.

### What remains before a robust deployment

| Action | Suggested owner | Indicative timeline |
|---|---|---|
| E7 — Calibration by category | Data Science | 1 week |
| Initial API design and deployment | Data Science + Technology | 2–3 weeks |
| Monitoring configuration | Data Science + Operations | 1–2 weeks |
| Controlled pilot in selected stores | Operations + Business | 4 weeks |

[⬆ Back to table of contents](#indice)

### 6.2 Recommended next steps

The recommended path for DSMarket would be:

1. **Approve this proposal** in a joint meeting between business, data and technology.
2. **Implement E7** (category-level calibration) as an immediate low-cost improvement.
3. **Develop the API** to expose the weekly order recommendation.
4. **Launch a controlled pilot in 2 stores**, one high-volume store and one with more volatile behavior.
5. **Review pilot results** after 4 weeks and decide whether to move to a broader deployment.

This sequence makes it possible to capture operational value early without requiring an overly complex infrastructure from the start.

[⬆ Back to table of contents](#indice)

## Executive memorandum — Response to Task 4

<div style="background-color:#264653;border-left:6px solid #264653;padding:14px;border-radius:10px">

**Subject:** Smart forecasting-based replenishment proposal for DSMarket

The conclusion of this task is that DSMarket already has a sufficient analytical foundation to pilot a smarter replenishment system than the current one.

The recommended solution does not consist of using the model prediction directly,
but of combining:

- **calibrated forecast**,  
- **safety stock**,  
- **available stock**,  
- and **continuous monitoring**.

From a technical perspective, **E2** was the best model.
From an operational perspective, the best alternative is **calibrated E2**,
because it reduces underforecasting bias and makes the replenishment decision safer.

The proposal is viable as a pilot and also identifies a clear evolution roadmap:

- finer calibration by category,
- model improvement for zero inflation,
- and deployment through an API and maintenance pipeline.

As a result, the recommendation is to move forward with a **controlled pilot**,
measure its real impact on stockouts and overstock,
and use that learning as the basis for a progressive rollout.

</div>

[⬆ Back to table of contents](#indice)

---

*Document prepared by Nicole Chen, Senior Data Scientist — DSMarket*  
*March 2025*